In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.linear_model import Ridge, Lasso
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [7]:
# Importing test data #
path = "C:/Users/Kolby.Porter/OneDrive - Hajoca Corporation/Documents/Custom Office Templates/MXL Custom Templates/Python"
Test_Data = pd.read_csv(path + "/test.csv")
Train_data = pd.read_csv(path + "/train.csv")

In [19]:
print(Train_data.head())
Train_data.shape


   Id  MSSubClass MSZoning  LotFrontage  LotArea Street Alley LotShape  \
0   1          60       RL         65.0     8450   Pave   NaN      Reg   
1   2          20       RL         80.0     9600   Pave   NaN      Reg   
2   3          60       RL         68.0    11250   Pave   NaN      IR1   
3   4          70       RL         60.0     9550   Pave   NaN      IR1   
4   5          60       RL         84.0    14260   Pave   NaN      IR1   

  LandContour Utilities  ... PoolArea PoolQC Fence MiscFeature MiscVal MoSold  \
0         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      2   
1         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      5   
2         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      9   
3         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      2   
4         Lvl    AllPub  ...        0    NaN   NaN         NaN       0     12   

  YrSold  SaleType  SaleCondition  SalePrice  
0   2008        WD   

(1460, 81)

In [23]:
## Exploring NA values ## 

Train_data.isna().sum().sort_values(ascending = False)
NAs = Train_data.isna()
NA_train_data = Train_data[NAs.any(axis = 1)]
NA_train_data.shape

(1460, 81)

In [35]:
Train_data.isna().sum().sort_values(ascending = False).head(10)

FireplaceQu     690
LotFrontage     259
GarageQual       81
GarageType       81
GarageCond       81
GarageYrBlt      81
GarageFinish     81
BsmtFinType2     38
BsmtExposure     38
BsmtQual         37
dtype: int64

**NA Heavy Features: Features missing from over 50% of the training data**
* PoolQC - Pool quality
* MiscFeature - Miscellaneous feature not covered in other categories
* Alley - Type of alley access
* Fence - Fence quality
* MasVnrType - Masonry veneer type <br>
**Honorable Mention:**
* FireplaceQu - Fireplace quality

In [34]:
### Handling NA Values ###
## Imputing 'None' for PoolQC where PoolArea is 0. This is because if there is no pool, there cannot be a pool quality rating. ##
Train_data['PoolQC'] = np.where(Train_data['PoolArea'] == 0, 'None', Train_data['PoolQC'])

## Imputing 'None' for MiscFeature where MiscVal is 0. This is because if there is no miscellaneous feature, there cannot be a miscellaneous feature rating. ##
Train_data['MiscFeature'] = np.where(Train_data['MiscVal'] == 0, 'None', Train_data['MiscFeature'])

## Imputing 'None' for MasVnrType where MasVNRArea is 0. This is because if there is no masonry veneer area, there cannot be a masonry veneer type. ##
Train_data['MasVnrType'] = np.where(Train_data['MasVnrArea'] == 0, 'None', Train_data['MasVnrType'])

## Imputing 'None' for alley access where there is no alley access. 
Train_data['Alley'] = np.where(Train_data['Alley'].isna(), 'None', Train_data['Alley'])

## Imputing 'None' for fence where there is no fence. 
Train_data['Fence'] = np.where(Train_data['Fence'].isna(), 'None', Train_data['Fence'])

In [ ]:
### Making the same imputations for the test data ###
## Imputing 'None' for PoolQC where PoolArea is 0. This is because if there is no pool, there cannot be a pool quality rating. ##
Test_Data['PoolQC'] = np.where(Test_Data['PoolArea'] == 0, 'None', Test_Data['PoolQC'])

## Imputing 'None' for MiscFeature where MiscVal is 0. This is because if there is no miscellaneous feature, there cannot be a miscellaneous feature rating. ##
Test_Data['MiscFeature'] = np.where(Test_Data['MiscVal'] == 0, 'None', Test_Data['MiscFeature'])

## Imputing 'None' for MasVnrType where MasVNRArea is 0. This is because if there is no masonry veneer area, there cannot be a masonry veneer type. ##
Test_Data['MasVnrType'] = np.where(Test_Data['MasVnrArea'] == 0, 'None', Test_Data['MasVnrType'])

## Imputing 'None' for alley access where there is no alley access. 
Test_Data['Alley'] = np.where(Test_Data['Alley'].isna(), 'None', Test_Data['Alley'])

## Imputing 'None' for fence where there is no fence. 
Test_Data['Fence'] = np.where(Test_Data['Fence'].isna(), 'None', Test_Data['Fence'])

In [60]:
## Splitting the data ##
train_set_x = Train_data.drop('SalePrice', axis = 1)
train_set_y = Train_data['SalePrice']

train_x, test_x, train_y, test_y = train_test_split(train_set_x, train_set_y, test_size = 0.25, random_state = 12)

In [ ]:
## Instantiating the linear model pipeline ##
num_features  = train_x.select_dtypes(include = ['number'] )
cat_features = train_x.select_dtypes(exclude = ['number'])

num_pipe_linear = Pipeline(steps = [
    ('scaler', StandardScaler()),
    ('imputer', SimpleImputer(strategy = 'median')),  
])
cat_pipe_linear = Pipeline(steps = [
    ('imputer', SimpleImputer(strategy = 'most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown = 'ignore', drop = 'first'))
])

preprocessor_linear = ColumnTransformer(transformers = [
    ('num', num_pipe_linear, num_features.columns),
    ('cat', cat_pipe_linear, cat_features.columns),
])


In [78]:
## Instantiating linear models ## 

ridge = Ridge()
lasso = Lasso()

pg = {
    'model__alpha': np.array([0.01, 0.1, 0.2, 0.25, 0.3])
}

ridge_pipe = Pipeline(steps = [
    ('preprocessor', preprocessor_linear),
    ('model', ridge)
])

lasso_pipe = Pipeline( steps = [
    ('preprocessor', preprocessor_linear),
    ('model', lasso)
])

kf = KFold(n_splits = 5, shuffle = True, random_state= 12)

GRidge = GridSearchCV(estimator = ridge_pipe, param_grid = pg, cv = kf, scoring= 'neg_root_mean_squared_error')
GLasso = GridSearchCV(estimator = lasso_pipe, param_grid = pg, cv = kf, scoring = 'neg_root_mean_squared_error')


In [81]:
## Fitting and predicting linear models ## 
GRidge.fit(train_x, train_y)
GRidge_pred = GRidge.predict(test_x)

GLasso.fit(train_x, train_y)
GLasso_pred = GLasso.predict(test_x)


c:\Users\Kolby.Porter\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [14, 16, 19, 22, 26, 29, 31, 41] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\Kolby.Porter\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [8, 10, 14] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\Kolby.Porter\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [9, 10, 15, 40] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\Kolby.Porter\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\preprocessing\_encoders.py

In [87]:
## Scoring the linear models ##
GRidge_rsme = mean_squared_error(test_y, GRidge_pred)
GLasso_rsme = mean_squared_error(test_y, GLasso_pred)

print(f'The RSME for the linear models are Ridge: {GRidge_rsme}, Lasso: {GLasso_rsme}')

GRidge_R2 = r2_score(test_y, GRidge_pred)
GLasso_R2 = r2_score(test_y, GLasso_pred)

print(f'The R2 for the linear models are Ridge: {GRidge_R2}, Lasso: {GLasso_R2}')

The RSME for the linear models are Ridge: 2472007099.360478, Lasso: 1545591568.7251713
The R2 for the linear models are Ridge: 0.6352323920443419, Lasso: 0.7719336083030799


In [97]:
## Instantiaating Tree Pipelines ##

num_pipe_tree = Pipeline( steps = [
    ('imputer', SimpleImputer(strategy = 'median'))
])

cat_pipe_tree = Pipeline( steps = [
    ('imputer', SimpleImputer(strategy = 'most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown= 'ignore'))
])

preprocessor_tree = ColumnTransformer(transformers = [
    ('num', num_pipe_tree, num_features.columns),
    ('cat', cat_pipe_tree, cat_features.columns)
])



In [98]:
## Instantiating Tree Based Models ## 
rf = RandomForestRegressor()
xgb = XGBRegressor()

rf_param_grid = {
    'model__n_estimators': [50, 100, 200],
    'model__max_depth': [None, 10, 20],
    'model__min_samples_split': [2,5,10]
}

xgb_param_grid = {
    'model__n_estimators': [50, 100, 200],
    'model__max_depth': [3, 6, 10],
    'model__learning_rate': [0.01, 0.1, 0.2]
}

rf_pipe = Pipeline(steps = [
    ('preprocessor', preprocessor_tree),
    ('model', rf)
])

xgb_pipe = Pipeline(steps = [
    ('preprocessor', preprocessor_tree),
    ('model', xgb)
])

kf2 = KFold(n_splits = 3, shuffle = True, random_state= 12)

GS_rf = GridSearchCV(estimator = rf_pipe, param_grid = rf_param_grid, cv = kf2, scoring= 'neg_root_mean_squared_error')
GS_xgb = GridSearchCV(estimator = xgb_pipe, param_grid = xgb_param_grid, cv = kf2, scoring= 'neg_root_mean_squared_error')


In [99]:
## Fitting and predicting tree based models ##
GS_rf.fit(train_x, train_y)
rf_pred = GS_rf.predict(test_x)

In [ ]:
GS_xgb.fit(train_x, train_y)
xgb_pred = GS_xgb.predict(test_x)

In [ ]:
## Scoring the Tree models ##
rf_rsme = mean_squared_error(test_y, rf_pred)
xgb_rsme = mean_squared_error(test_y, xgb_pred)

print(f'The RSME for the tree models are RF: {rf_rsme}, XGB: {xgb_rsme}')

rf_R2 = r2_score(test_y, rf_pred)
xgb_R2 = r2_score(test_y, xgb_pred)

print(f'The R2 for the tree models are RF: {rf_R2}, XGB: {xgb_R2}')

In [100]:
## Summary model performance ##
model_performance = pd.DataFrame({
    'Model': ['Ridge', 'Lasso', 'Random Forest', 'XGBoost'],
    'RSME': [GRidge_rsme, GLasso_rsme, rf_rsme, xgb_rsme],
    'R2': [GRidge_R2, GLasso_R2, rf_R2, xgb_R2]
})

NameError: name 'rf_rsme' is not defined